# Experiment: raportul fraude : legitime la antrenare (VQC oficial, fără zgomot)

**Fără zgomot, p = 0 — evaluare pe validation.**

## Scop
Vrem să aflăm dacă **mai multe tranzacții legitime la antrenare** îmbunătățesc detectarea fraudelor de către VQC-ul din Task 1 și **cum se schimbă costul** estimat pe hardware. Comparăm trei raporturi FRAUDE : LEGITIME în setul de antrenare:

| raport | fraude | legitime | total (split oficial) |
|---|---|---|---|
| 1:1 | 295 | 295 | 590 |
| 1:3 | 295 | 885 | 1.180 |
| 1:5 | 295 | 1.475 | 1.770 |

Dimensiunile sunt generate în cod din numărul efectiv de fraude din training și verificate.

## Ipoteză (de testat, nu presupusă)
- **H1:** mai multe legitime arată modelului mai bine variația clasei majoritare, ceea ce ar putea crește precision și AUPRC pe validation (unde rata de fraudă e 0,17%).
- **H0 / alternativă:** VQC-ul cu 32 de parametri nu profită de date în plus, sau dezechilibrul mai mare împinge scorurile spre „legitim” și scade recall-ul.

Nu presupunem că vreo variantă e mai bună. Experimentul doar măsoară.

## Ce schimbăm și ce ținem fix
- **Se schimbă:** doar numărul de legitime din training (subseturi imbricate: 1:1 ⊂ 1:3 ⊂ 1:5, pentru fiecare seed).
- **Fix:** splitul, encoderul (fitat pe subsetul 1:1 al seed-ului), circuitul, ponderile inițiale, optimizer-ul, learning rate-ul, batch size-ul, numărul de epoci și protocolul de alegere a epocii și a pragului.
- **Nu folosim testul** pentru nimic în afară de verificarea indicilor și a numărului de clase. Nicio transformare și niciun scor pe test.
- Nu există încă referință clasică (logistic regression). Va fi adăugată separat, pentru evaluarea finală.

## 1. Configurație și imports

`SMOKE_TEST = True` rulează **un seed și 2 epoci** pentru toate cele trei raporturi: este doar o verificare tehnică a pipeline-ului (rezultatele în `results/result_exp1/test1_without_noise/`). **Setarea curentă este `SMOKE_TEST = False`: experimentul complet** (seeds `[0, 1, 2, 3, 4]`, 30 de epoci, rezultatele în `results/result_exp1/test1_full_without_noise/`).

**Seeds.** `set_seed` fixează numpy și torch. Amestecarea exemplelor folosește un `torch.Generator` dedicat, inițializat cu seed-ul rulării. PennyLane rulează pe `default.qubit` cu `shots=None` (valori de așteptare exacte), deci simulatorul nu are aleatorietate de controlat.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ---- mod de rulare ----
SMOKE_TEST = False            # True: verificare tehnică (1 seed, 2 epoci). False: experimentul complet.

# ---- protocol comun (identic cu notebookul oficial) ----
SPLIT_SEED = 42               # seed-ul splitului comun 60/20/20
N_QUBITS = 4                  # o componentă PCA per qubit
FEATURE_MAP_REPS = 2          # repetări ale ZZ feature map
ANSATZ_REPS = 3               # straturi de entanglement ale ansatz-ului: (3 + 1) * 4 * 2 = 32 parametri
BATCH_SIZE = 32
LEARNING_RATE = 0.05
SHOTS_PER_EXECUTION = 1024    # folosit doar în estimarea nominală a costului
NATIVE_GATES = {"RX", "RY", "RZ", "Rot", "Hadamard", "CNOT", "CZ"}   # doar pentru numărarea porților

# ---- experiment ----
RATIOS = [1, 3, 5]            # legitime per fraudă în training; 1:1 trebuie să fie primul
EXTRA_LEGIT_STREAM = 1        # legitimele suplimentare: np.random.default_rng((seed, EXTRA_LEGIT_STREAM))
EXPERIMENT_NAME = "ratio_experiment_p0"

if SMOKE_TEST:
    MODE, SEEDS, EPOCHS = "smoke", [0], 2
else:
    MODE, SEEDS, EPOCHS = "full", [0, 1, 2, 3, 4], 30

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
RESULTS_SUBDIR = {"smoke": "test1_without_noise", "full": "test1_full_without_noise"}[MODE]   # directoarele redenumite; "mode" din config.json rămâne smoke / full
RESULTS_DIR = REPO_ROOT / "results" / "result_exp1" / RESULTS_SUBDIR   # EXPERIMENT_NAME rămâne doar eticheta din config.json
TABLE_LABEL = "Fără zgomot, p = 0 — evaluare pe validation"

LEGIT_COLOR, FRAUD_COLOR, SEED_DOT_COLOR = "#2a78d6", "#eb6834", "#8a8f98"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SPLIT_SEED)
print(f"mod: {MODE}   seeds: {SEEDS}   epochs: {EPOCHS}   raporturi: {['1:%d' % r for r in RATIOS]}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__)
print("repo:", REPO_ROOT)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Încărcarea datelor și verificări

Citim `data/creditcard.csv` (identic cu OpenML 42175), ca în notebookul oficial. **Nu** eliminăm duplicate, outlieri sau tranzacții cu `Amount = 0`, și nu reordonăm și nu resetăm indicii: splitul depinde de ordinea rândurilor.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    data_source = "data/creditcard.csv"
else:
    df = fetch_openml(data_id=42175, as_frame=True, parser="auto").frame
    data_source = "OpenML 42175"
df["Class"] = df["Class"].astype(int)

assert df.shape == (284807, 31), f"forma neașteptată: {df.shape}"
assert "Time" in df.columns, "lipsește Time: probabil OpenML 1597, nu 42175"
assert df.index.equals(pd.RangeIndex(len(df))), "indexul trebuie să fie 0..N-1, în ordinea fișierului"
n_legit_all, n_fraud_all = df["Class"].value_counts().sort_index().tolist()
assert (n_legit_all, n_fraud_all) == (284315, 492)

print(f"sursă: {data_source}   rânduri: {len(df):,}   legitime: {n_legit_all:,}   fraude: {n_fraud_all}")
print(f"rânduri duplicate în dataset (informativ, NU se elimină): {df.duplicated().sum():,}")
print(f"tranzacții cu Amount = 0 (informativ, NU se elimină): {(df['Amount'] == 0).sum():,}")

## 3. Splitul comun

Dacă există `split_indices.npz` în rădăcina repo-ului, îl folosim doar dacă e identic cu reconstrucția oficială. Altfel reconstruim splitul exact ca în Partea 4 a notebookului: dataset integral, `SEED = 42`, split stratificat în doi pași (test 20%, apoi validation 25% din rest). Splitul este același pentru toate seed-urile și toate raporturile.

Din test păstrăm **doar indicii și etichetele**, pentru verificările de integritate. Matricea de features a testului este ștearsă din memorie, ca să nu poată fi transformată sau scorată din greșeală.

In [ ]:
X = df.drop(columns="Class")
y = df["Class"].to_numpy()
X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SPLIT_SEED)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.25, stratify=y_trval, random_state=SPLIT_SEED)
rebuilt = {"train": X_train.index.to_numpy(), "val": X_val.index.to_numpy(), "test": X_test.index.to_numpy()}

SPLIT_PATH = REPO_ROOT / "split_indices.npz"
if SPLIT_PATH.exists():
    saved = np.load(SPLIT_PATH)
    if not all(np.array_equal(saved[k], rebuilt[k]) for k in rebuilt):
        raise ValueError("split_indices.npz nu corespunde splitului oficial reconstruit: verifică fișierul înainte de a continua")
    split_source = "split_indices.npz (identic cu reconstrucția oficială)"
else:
    split_source = "reconstruit ca în notebookul oficial (split_indices.npz lipsește)"

train_idx, val_idx, test_idx = rebuilt["train"], rebuilt["val"], rebuilt["test"]
n_test_rows, n_test_frauds = len(y_test), int(y_test.sum())
del X_test, X_trval, y_trval        # testul nu se folosește în acest experiment

# verificări de integritate
for name, idx in rebuilt.items():
    assert len(np.unique(idx)) == len(idx), f"indici repetați în {name}"
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx)), "seturile se suprapun"
assert len(train_idx) + len(val_idx) + len(test_idx) == len(df) and set(np.concatenate(list(rebuilt.values()))) == set(df.index), "splitul nu acoperă tot datasetul"
assert (int(y_train.sum()), int(y_val.sum()), n_test_frauds) == (295, 99, 98), "numărul de fraude trebuie să fie 295/99/98"
assert (y[train_idx] == y_train).all() and (y[val_idx] == y_val).all()
assert n_test_rows == 56962

print("split:", split_source)
display(pd.DataFrame(
    {"rânduri": [len(train_idx), len(val_idx), n_test_rows], "fraude": [int(y_train.sum()), int(y_val.sum()), n_test_frauds],
     "rată fraudă": [f"{y_train.mean():.3%}", f"{y_val.mean():.3%}", f"{n_test_frauds / n_test_rows:.3%}"]},
    index=["train", "validation", "test (doar indici și clase)"],
))
print("✅ split: disjunct, acoperă tot datasetul, fraude 295/99/98")

## 4. Funcțiile din notebookul oficial

Notebookul oficial local este versiunea cu exerciții (`FILL_IN`). Funcțiile de mai jos sunt copiate din el, cu golurile completate, și verificate cu aceleași verificări ✅ din notebook (Check 4.2, 9.1, 6.1, 6.2, 6.3, 8.1). Nu folosim `%run` pe notebookul oficial.

In [ ]:
def undersample(X_part, y_part, seed):
    rng = np.random.default_rng(seed)
    fraud = np.flatnonzero(y_part == 1)
    legit = rng.choice(np.flatnonzero(y_part == 0), size=len(fraud), replace=False)   # as many distinct legitimate positions as there are frauds
    keep = rng.permutation(np.concatenate([fraud, legit]))   # shuffle, so that frauds are not all first
    return X_part.iloc[keep], y_part[keep]


class AngleEncoder:
    raw_columns = ["Time", "Amount"]

    def __init__(self, n_components, seed):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=seed)
        self.angles = MinMaxScaler(feature_range=(0, np.pi))

    def _scaled(self, frame):
        out = frame.copy()
        out[self.raw_columns] = self.scaler.transform(frame[self.raw_columns])
        return out.to_numpy()

    def fit(self, frame):
        self.scaler.fit(frame[self.raw_columns])
        self.angles.fit(self.pca.fit_transform(self._scaled(frame)))
        return self

    def transform(self, frame):
        return np.clip(self.angles.transform(self.pca.transform(self._scaled(frame))), 0, np.pi)


def choose_threshold(y_true, scores):
    """The threshold that maximises F1 on (y_true, scores)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(y_true, scores, threshold):
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "AUPRC": average_precision_score(y_true, scores),
        "AUROC": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "F1": f1_score(y_true, predicted, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predicted),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }

In [ ]:
# ✅ Check 4.2 (din notebook): undersample pe training, cu seed-ul oficial
X_bal_check, y_bal_check = undersample(X_train, y_train, SPLIT_SEED)
assert len(y_bal_check) == 2 * y_train.sum() and y_bal_check.sum() == y_train.sum(), "keep every training fraud and as many legitimate rows"
assert X_bal_check.index.is_unique, "draw without replacement: no row may appear twice"
assert set(X_bal_check.index) <= set(X_train.index), "balance the training set only"

# ✅ Check 9.1 (din notebook): pragul maximizează F1
rng_check = np.random.default_rng(3)
y_c = rng_check.integers(0, 2, 300)
s_c = np.clip(0.3 * y_c + rng_check.random(300), 0, 1)
best_f1 = max(f1_score(y_c, (s_c >= t).astype(int)) for t in np.unique(s_c))
assert np.isclose(f1_score(y_c, (s_c >= choose_threshold(y_c, s_c)).astype(int)), best_f1), "the threshold should maximise F1"
print("✅ Check 4.2 și 9.1: trecute")

## 5. Subseturi imbricate de training

Pentru fiecare seed:
1. **1:1** este exact ieșirea funcției oficiale `undersample(X_train, y_train, seed)`: toate cele 295 de fraude + 295 de legitime, în ordinea amestecată de funcție. Nu o modificăm.
2. Din legitimele **rămase** (cele care nu sunt în 1:1) tragem, **fără înlocuire**, cu un generator separat `np.random.default_rng((seed, 1))`, o listă de `(5 − 1) × 295` legitime suplimentare.
3. **1:3** = 1:1 + primele `2 × 295` legitime suplimentare. **1:5** = 1:1 + primele `4 × 295`. Astfel 1:1 ⊂ 1:3 ⊂ 1:5.

Lucrăm cu **poziții** în `X_train` (nu cu etichetele indexului), apoi verificăm că toate exemplele aparțin trainingului, că nu se repetă niciun indice într-un subset și că imbricarea e respectată. Rândurile identice cu indici diferiți rămân în date: sunt raportate informativ, nu eliminate.

In [ ]:
assert RATIOS[0] == 1 and RATIOS == sorted(RATIOS)


def nested_training_subsets(seed):
    """ratio -> pozițiile din X_train ale subsetului de antrenare, cu 1:1 = undersample(...) oficial."""
    X_11, _ = undersample(X_train, y_train, seed)
    pos_11 = X_train.index.get_indexer(X_11.index)
    n_fraud = int(y_train.sum())
    legit_11 = pos_11[y_train[pos_11] == 0]
    remaining = np.setdiff1d(np.flatnonzero(y_train == 0), legit_11)
    rng = np.random.default_rng((seed, EXTRA_LEGIT_STREAM))
    extra = rng.choice(remaining, size=(max(RATIOS) - 1) * n_fraud, replace=False)
    return {r: pos_11 if r == 1 else np.concatenate([pos_11, extra[:(r - 1) * n_fraud]]) for r in RATIOS}


def check_subsets(seed, subsets):
    n_fraud = int(y_train.sum())
    X_11, y_11 = undersample(X_train, y_train, seed)
    assert X_train.index[subsets[1]].equals(X_11.index) and (y_train[subsets[1]] == y_11).all(), "1:1 trebuie să fie exact undersample(...)"
    legit_sets, rows = {}, []
    for r, pos in subsets.items():
        labels = X_train.index[pos]
        assert (pos >= 0).all() and len(np.unique(pos)) == len(pos), f"1:{r}: poziții invalide sau repetate"
        assert set(labels) <= set(train_idx) and not set(labels) & (set(val_idx) | set(test_idx)), f"1:{r}: exemple din afara trainingului"
        assert set(np.flatnonzero(y_train == 1)) <= set(pos), f"1:{r}: lipsesc fraude din training"
        n_f, n_l = int(y_train[pos].sum()), int((y_train[pos] == 0).sum())
        assert (n_f, n_l) == (n_fraud, r * n_fraud), f"1:{r}: {n_f} fraude, {n_l} legitime"
        legit_sets[r] = set(pos[y_train[pos] == 0])
        rows.append({"seed": seed, "raport": f"1:{r}", "fraude": n_f, "legitime": n_l, "total": len(pos),
                     "rânduri identice (informativ)": int(X_train.iloc[pos].duplicated().sum())})
    for small, big in zip(RATIOS[:-1], RATIOS[1:]):
        assert legit_sets[small] <= legit_sets[big], f"legitimele din 1:{small} nu sunt incluse în 1:{big}"
        assert (subsets[big][:len(subsets[small])] == subsets[small]).all()
    return rows


n_fraud_train = int(y_train.sum())
expected_sizes = {r: (n_fraud_train, r * n_fraud_train, (1 + r) * n_fraud_train) for r in RATIOS}
print("dimensiuni generate din numărul de fraude din training:", {f"1:{r}": v for r, v in expected_sizes.items()})
assert expected_sizes == {1: (295, 295, 590), 3: (295, 885, 1180), 5: (295, 1475, 1770)}

subset_rows = []
for seed in SEEDS:
    subset_rows += check_subsets(seed, nested_training_subsets(seed))
display(pd.DataFrame(subset_rows))
print("✅ subseturi: 1:1 = undersample oficial, fără repetări, doar training, imbricate 1:1 ⊂ 1:3 ⊂ 1:5")

## 6. Encoder fix per seed

**Alegere experimentală intenționată.** Pentru fiecare seed, `AngleEncoder` (StandardScaler pe `Time` și `Amount` → PCA la 4 componente → `MinMaxScaler` la $[0, \pi]$ → clipping la $[0, \pi]$) se fitează **o singură dată, doar pe subsetul 1:1** al acelui seed, și se aplică neschimbat pe 1:1, 1:3, 1:5 și validation.

Motivul: vrem să schimbăm **doar datele cu care se antrenează clasificatorul**, păstrând **reprezentarea fixă** între raporturi. Dacă am reajusta encoderul pe 1:3 și 1:5, s-ar schimba și direcțiile PCA, și scalarea unghiurilor, deci nu am mai ști dacă o diferență vine de la numărul de legitime sau de la altă reprezentare.

Consecințe de știut:
- encoderul este fitat tot numai pe training (fără leakage), dar poate diferi între seeds, pentru că subsetul 1:1 diferă;
- legitimele suplimentare din 1:3 și 1:5 nu au contribuit la min/max-ul unghiurilor, așa că unele valori pot ieși din $[0, \pi]$ și sunt **clipped**, exact ca validarea. Fracția lor este raportată mai jos;
- encoderul nu este optim pentru 1:3 și 1:5. Un experiment separat ar putea compara „encoder fitat pe fiecare raport”.

In [ ]:
def encode_seed(seed, subsets):
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_train.iloc[subsets[1]])   # fit DOAR pe 1:1
    Z = {r: encoder.transform(X_train.iloc[pos]) for r, pos in subsets.items()}
    return encoder, Z, encoder.transform(X_val)


def clipped_fraction(encoder, frame):
    raw = encoder.angles.transform(encoder.pca.transform(encoder._scaled(frame)))
    return float(((raw < 0) | (raw > np.pi)).any(axis=1).mean())


def check_encoding(seed, subsets, encoder, Z, Z_val):
    assert encoder.pca.n_samples_ == len(subsets[1]), "encoderul trebuie fitat doar pe subsetul 1:1"
    assert np.allclose(Z[1].min(axis=0), 0) and np.allclose(Z[1].max(axis=0), np.pi), "unghiurile 1:1 trebuie să acopere exact [0, pi]"
    for r in RATIOS:
        assert Z[r].shape == (len(subsets[r]), N_QUBITS) and Z[r].min() >= 0 and Z[r].max() <= np.pi
        assert np.array_equal(Z[r][:len(subsets[1])], Z[1]), "același encoder: partea 1:1 trebuie să fie identică"
    assert Z_val.shape == (len(val_idx), N_QUBITS) and Z_val.min() >= 0 and Z_val.max() <= np.pi
    return {"seed": seed, "varianță explicată (4 PC)": round(float(encoder.pca.explained_variance_ratio_.sum()), 3),
            **{f"rânduri clipped 1:{r}": f"{clipped_fraction(encoder, X_train.iloc[subsets[r]]):.2%}" for r in RATIOS},
            "rânduri clipped validation": f"{clipped_fraction(encoder, X_val):.2%}"}


encoding_rows = []
for seed in SEEDS:
    subsets = nested_training_subsets(seed)
    encoding_rows.append(check_encoding(seed, subsets, *encode_seed(seed, subsets)))
display(pd.DataFrame(encoding_rows))
print("✅ encoder: fitat doar pe 1:1, reutilizat identic pentru 1:3, 1:5 și validation; unghiuri în [0, pi]")

## 7. VQC-ul original și funcțiile de antrenare

Circuitul este cel din Partea 6 a notebookului: 4 qubiți, ZZ feature map cu 2 repetări, ansatz EfficientSU2-style cu 3 straturi de entanglement și stratul final de rotații (32 de parametri), readout $(1 - \langle Z_0\rangle)/2$, `default.qubit` fără zgomot, `shots=None`, backprop în PyTorch.

In [ ]:
def feature_map(x):
    """ZZ feature map. x holds N_QUBITS angles in [0, pi], or a batch of them (shape (..., N_QUBITS))."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[..., i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def ansatz(weights):
    """EfficientSU2-style ansatz; weights has shape (ANSATZ_REPS + 1, N_QUBITS, 2)."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def vqc(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


device = qml.device("default.qubit", wires=N_QUBITS)   # shots=None: valori de așteptare exacte
circuit = qml.QNode(vqc, device, interface="torch", diff_method="backprop")


def fraud_probability(x, weights):
    return (1 - circuit(x, weights)) / 2


def init_weights(seed):
    generator = torch.Generator().manual_seed(seed)
    return torch.nn.Parameter(2 * np.pi * torch.rand(ANSATZ_REPS + 1, N_QUBITS, 2, generator=generator, dtype=torch.float64))


def predict(Z, weights, model=None, chunk=20000):
    model = model or fraud_probability
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([model(X_t[k:k + chunk], weights) for k in range(0, len(X_t), chunk)]).numpy()

In [ ]:
# ✅ Checks 6.1, 6.2, 6.3, 8.1 (din notebook), pe encodarea 1:1 a primului seed
dev4 = qml.device("default.qubit", wires=N_QUBITS)


def state_of(circuit_fn, *args):
    def full():
        circuit_fn(*args)
        return qml.state()
    return qml.QNode(full, dev4)()


def gate_counts(circuit_fn, *args):
    def full():
        circuit_fn(*args)
        return qml.state()
    return qml.specs(qml.transforms.decompose(qml.QNode(full, dev4), gate_set=NATIVE_GATES))()["resources"].gate_types


def feature_map_reference(x):
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.IsingZZ(2 * (np.pi - x[i]) * (np.pi - x[j]), wires=[i, j])


_, Z_check, _ = encode_seed(SEEDS[0], nested_training_subsets(SEEDS[0]))
Z_check = Z_check[1]

for x_check in Z_check[:3]:   # 6.1
    overlap = abs(np.vdot(state_of(feature_map_reference, x_check), state_of(feature_map, x_check))) ** 2
    assert np.isclose(overlap, 1.0), f"feature map: overlap {overlap:.3f} instead of 1"

w_check = torch.linspace(0, 3, (ANSATZ_REPS + 1) * N_QUBITS * 2, dtype=torch.float64).reshape(ANSATZ_REPS + 1, N_QUBITS, 2)   # 6.2
counts = gate_counts(ansatz, w_check)
n_rotations, n_cnots = (ANSATZ_REPS + 1) * N_QUBITS, ANSATZ_REPS * (N_QUBITS - 1)
assert counts.get("RY", 0) == n_rotations and counts.get("RZ", 0) == n_rotations and counts.get("CNOT", 0) == n_cnots
probabilities = np.abs(qml.math.to_numpy(state_of(ansatz, w_check))) ** 2
assert np.allclose(probabilities[[0, 5, 10, 15]], [0.049016, 0.01134, 0.037841, 0.048166], atol=1e-5), "ansatz: wrong order or angles"


@qml.qnode(dev4, interface="torch")   # 6.3
def qubit0_probabilities(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.probs(wires=0)


example_x, example_w = torch.tensor(Z_check[:5]), init_weights(SEEDS[0])
with torch.no_grad():
    assert torch.allclose(fraud_probability(example_x, example_w), qubit0_probabilities(example_x, example_w)[:, 1])

N_PARAMS = example_w.numel()   # 8.1
EXECUTIONS_PER_SAMPLE = 1 + 2 * N_PARAMS
assert N_PARAMS == 32 and EXECUTIONS_PER_SAMPLE == 65

full_counts = gate_counts(vqc, Z_check[0], example_w)
assert sum(full_counts.values()) == 93 and full_counts["CNOT"] == 33, dict(full_counts)
assert torch.equal(init_weights(SEEDS[0]), init_weights(SEEDS[0])), "inițializarea trebuie să fie deterministă"

# amestecarea cu generator dedicat reproduce exact torch.randperm global după set_seed (ca în notebook)
set_seed(SEEDS[0]); perm_global = torch.randperm(590)
assert torch.equal(perm_global, torch.randperm(590, generator=torch.Generator().manual_seed(SEEDS[0])))

print(f"VQC: {N_QUBITS} qubiți, {sum(full_counts.values())} porți ({full_counts['CNOT']} CNOT), {N_PARAMS} parametri, {EXECUTIONS_PER_SAMPLE} execuții/exemplu (parameter shift nominal)")
print("✅ Checks 6.1, 6.2, 6.3, 8.1: trecute")

### Bucla de antrenare

Aceeași buclă ca `train_vqc` din Partea 7 (Adam, BCE, batch 32, scorarea validării după fiecare epocă, păstrarea ponderilor epocii cu cel mai bun AUPRC, primul maxim în caz de egalitate), cu trei diferențe care nu schimbă algoritmul:
- primește **o copie independentă** a ponderilor inițiale, deci toate raporturile aceluiași seed pornesc din exact aceleași valori, fiecare cu un optimizer Adam nou;
- amestecarea folosește un `torch.Generator` dedicat, inițializat cu seed-ul. Verificarea de mai sus arată că dă aceeași permutare ca `torch.randperm` global după `set_seed(seed)`, deci 1:1 reproduce procedura oficială;
- măsoară timpul și numără pașii Adam și evaluările complete pe validation.

**Definiția timpului de antrenare** (`timp antrenare [s]`): timpul de perete (`time.perf_counter`) al întregii bucle `train_vqc_run`, adică pașii de optimizare (forward, backward, Adam) din toate epocile **plus** scorarea completă a validării și calculul AUPRC după fiecare epocă. Nu include încărcarea datelor, encodarea și evaluarea finală. Componentele sunt salvate și separat (`timp pași [s]`, `timp validare, toate epocile [s]`), iar evaluarea finală are coloana ei. Timpul depinde de mașină și de încărcarea ei, deci e orientativ.

In [ ]:
def train_vqc_run(Z_tr, y_tr, Z_valid, y_valid, initial_weights, seed, epochs=EPOCHS):
    set_seed(seed)
    shuffle_generator = torch.Generator().manual_seed(seed)
    weights = torch.nn.Parameter(initial_weights.detach().clone())   # copie independentă
    optimizer = torch.optim.Adam([weights], lr=LEARNING_RATE)          # optimizer nou
    loss_fn = torch.nn.BCELoss()
    X_t = torch.tensor(Z_tr)
    y_t = torch.tensor(y_tr, dtype=torch.float64)
    history, best_ap, best_weights, best_epoch = [], -1.0, None, None
    n_steps, n_val_evals, time_steps, time_val = 0, 0, 0.0, 0.0
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()
        order = torch.randperm(len(X_t), generator=shuffle_generator)
        losses = []
        for k in range(0, len(X_t), BATCH_SIZE):
            idx = order[k:k + BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(fraud_probability(X_t[idx], weights), y_t[idx])
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            n_steps += 1
        t1 = time.perf_counter()
        val_ap = average_precision_score(y_valid, predict(Z_valid, weights))
        n_val_evals += 1
        t2 = time.perf_counter()
        time_steps, time_val = time_steps + (t1 - t0), time_val + (t2 - t1)
        history.append({"epoch": epoch, "train loss": float(np.mean(losses)), "validation AUPRC": float(val_ap)})
        if val_ap > best_ap:
            best_ap, best_weights, best_epoch = val_ap, weights.detach().clone(), epoch
    return {
        "weights": best_weights, "history": history, "best_epoch": best_epoch, "best_val_ap": float(best_ap),
        "epochs_run": len(history), "optimizer_steps": n_steps, "val_evals": n_val_evals,
        "time_train_s": time.perf_counter() - start, "time_steps_s": time_steps, "time_val_epochs_s": time_val,
    }

### Evidența costului (estimare nominală pe hardware)

Ca în Partea 8 a notebookului, cu $P = 32$ parametri antrenabili:
- **training:** epoci efectiv executate × exemple de training × $(1 + 2P)$ = epoci × N × 65;
- **validation, toate epocile:** număr de evaluări complete (una după fiecare epocă) × exemple de validation (56.962), câte o execuție per tranzacție; valoarea e **suma pe toate epocile** unei rulări;
- **validation finală:** scorarea suplimentară a validării cu ponderile epocii alese (pentru prag și metrici), raportată **separat**: 1 × 56.962 per rulare;
- **shots:** execuții × 1024.

Se contabilizează **toate epocile executate**, nu doar epoca aleasă. Formula este **nominală**: pe hardware, parameter shift poate omite parametrii care nu influențează qubitul 0 (notebookul arată 41 de execuții reale per exemplu în loc de 65, cu `qml.Tracker`). Aceste numere **nu** sunt execuțiile reale ale simulatorului: aici simulatorul folosește backprop, `shots=None` și broadcasting, deci execuția „reală” nu are corespondent direct.

**Buget inegal.** Un număr egal de epoci nu înseamnă buget egal: la 30 de epoci, 1:5 procesează de 3 ori mai multe exemple decât 1:1 și face de ~3 ori mai mulți pași Adam (56 față de 19 per epocă). Experimentul compară raporturile **la număr fix de epoci, nu la buget fix**.

In [ ]:
def cost_ledger(n_train, epochs_run, val_evals, n_val):
    train_exec = epochs_run * n_train * EXECUTIONS_PER_SAMPLE
    val_epoch_exec = val_evals * n_val
    val_final_exec = 1 * n_val
    return {
        "exec training": train_exec, "exec validation, toate epocile": val_epoch_exec, "exec validation finală": val_final_exec,
        "shots training": train_exec * SHOTS_PER_EXECUTION, "shots validation, toate epocile": val_epoch_exec * SHOTS_PER_EXECUTION,
        "shots validation finală": val_final_exec * SHOTS_PER_EXECUTION,
    }


# verificare față de numerele din enunț: 30 epoci × 590 × 65 = 1.150.500; 30 × 56.962 = 1.708.860
_ref = cost_ledger(590, 30, 30, len(val_idx))
assert (_ref["exec training"], _ref["exec validation, toate epocile"]) == (1_150_500, 1_708_860)
print("✅ ledger: reproduce 1.150.500 execuții de training și 1.708.860 de validare din enunț (1:1, 30 epoci)")

## 8. Experimentul

Pentru fiecare seed: subseturi imbricate → encoder fitat pe 1:1 → aceleași ponderi inițiale `init_weights(seed)` → câte o rulare per raport. Pentru fiecare rulare:
1. antrenare 30 de epoci (2 în smoke test), cu păstrarea ponderilor epocii cu cel mai bun AUPRC pe validation;
2. scorarea validării cu aceste ponderi (verificăm că AUPRC coincide cu cel din buclă);
3. pragul care maximizează F1 pe validation (`choose_threshold`), apoi metricile la acest prag (`evaluate`).

**Salvare incrementală.** Fiecare rulare terminată se adaugă în `results/result_exp1/<test1_without_noise|test1_full_without_noise>/runs.jsonl` (istoricul pe epoci în `histories.jsonl`), iar configurația în `config.json`. La o nouă rulare cu aceeași configurație, combinațiile deja terminate sunt încărcate, nu reantrenate. Dacă configurația diferă, notebookul se oprește în loc să amestece sau să suprascrie rezultatele: folosește alt `EXPERIMENT_NAME`. Smoke testul și experimentul complet stau în directoare separate.

In [ ]:
def to_builtin(value):
    if isinstance(value, dict):
        return {k: to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


config = {
    "experiment": EXPERIMENT_NAME, "mode": MODE, "seeds": SEEDS, "epochs": EPOCHS, "ratios_legit_per_fraud": RATIOS,
    "noise": "none (p = 0), default.qubit, shots=None, diff_method=backprop",
    "evaluation_set": "validation (test not used)", "split_seed": SPLIT_SEED, "split_source": split_source,
    "n_train": len(train_idx), "n_val": len(val_idx), "n_test": n_test_rows, "frauds_train_val_test": [n_fraud_train, int(y_val.sum()), n_test_frauds],
    "subsets": "1:1 = official undersample(X_train, y_train, seed); extra legit without replacement from the rest, rng = np.random.default_rng((seed, %d)); nested" % EXTRA_LEGIT_STREAM,
    "encoder": "AngleEncoder(4, seed) fitted on the 1:1 subset of each seed, reused for 1:3, 1:5, validation",
    "n_qubits": N_QUBITS, "feature_map_reps": FEATURE_MAP_REPS, "ansatz_reps": ANSATZ_REPS, "n_params": N_PARAMS,
    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "optimizer": "Adam", "loss": "BCELoss",
    "init": "init_weights(seed), same copy for every ratio", "shuffle": "torch.Generator().manual_seed(seed)",
    "epoch_selection": "best validation AUPRC (average_precision_score)", "threshold": "max F1 on validation (choose_threshold)",
    "cost_formula": "train = epochs_run * n_train * (1 + 2P); validation = val_evals * n_val (+ 1 final pass, separate); shots = exec * 1024",
    "shots_per_execution": SHOTS_PER_EXECUTION, "data_source": data_source,
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH, RUNS_PATH, HIST_PATH = RESULTS_DIR / "config.json", RESULTS_DIR / "runs.jsonl", RESULTS_DIR / "histories.jsonl"
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    ignore = {"versions", "split_source"}   # split_source descrie de unde vine splitul (fișier sau reconstrucție), nu un parametru
    current = json.loads(json.dumps(to_builtin(config)))
    if {k: v for k, v in previous.items() if k not in ignore} != {k: v for k, v in current.items() if k not in ignore}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație. Nu suprascriu: schimbă EXPERIMENT_NAME sau mută directorul.")
else:
    CONFIG_PATH.write_text(json.dumps(to_builtin(config), indent=2, ensure_ascii=False), encoding="utf-8")


RENAMED_KEYS = {   # etichete vechi (smoke test inițial): valorile erau deja sume pe toate epocile
    "exec validation per epocă": "exec validation, toate epocile",
    "shots validation per epocă": "shots validation, toate epocile",
    "timp validare per epocă [s]": "timp validare, toate epocile [s]",
}


def load_jsonl(path, required=False):
    if not path.exists():
        if required:
            raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
        return []
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return [{RENAMED_KEYS.get(k, k): v for k, v in r.items()} for r in records]


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")


done = {(r["seed"], r["ratio"]) for r in load_jsonl(RUNS_PATH)}
print(f"config: {CONFIG_PATH.relative_to(REPO_ROOT)}   rulări deja salvate: {len(done)}")

In [ ]:
experiment_start = time.perf_counter()
for seed in SEEDS:
    subsets = nested_training_subsets(seed)
    check_subsets(seed, subsets)
    encoder, Z, Z_val = encode_seed(seed, subsets)
    check_encoding(seed, subsets, encoder, Z, Z_val)
    initial_weights = init_weights(seed).detach().clone()
    for r in RATIOS:
        if (seed, r) in done:
            print(f"seed {seed}  1:{r}  deja salvat, sărit")
            continue
        y_tr = y_train[subsets[r]]
        run = train_vqc_run(Z[r], y_tr, Z_val, y_val, initial_weights, seed)
        assert torch.equal(initial_weights, init_weights(seed).detach()), "ponderile inițiale au fost modificate"

        t0 = time.perf_counter()
        val_scores = predict(Z_val, run["weights"])
        threshold = choose_threshold(y_val, val_scores)
        metrics = evaluate(y_val, val_scores, threshold)
        time_final = time.perf_counter() - t0
        assert np.isclose(metrics["AUPRC"], run["best_val_ap"]), "AUPRC final diferă de cel al epocii alese"

        record = {
            "seed": seed, "ratio": r, "raport": f"1:{r}",
            "fraude train": int(y_tr.sum()), "legitime train": int((y_tr == 0).sum()), "total train": len(y_tr),
            "val AUPRC": metrics["AUPRC"], "val AUROC": metrics["AUROC"], "prag": float(threshold),
            "precision": metrics["precision"], "recall": metrics["recall"], "F1": metrics["F1"], "MCC": metrics["MCC"],
            "TP": metrics["TP"], "FP": metrics["FP"], "FN": metrics["FN"], "TN": metrics["TN"],
            "epoca aleasă": run["best_epoch"], "epoci executate": run["epochs_run"], "pași Adam": run["optimizer_steps"],
            "evaluări validation (epoci)": run["val_evals"],
            "timp antrenare [s]": run["time_train_s"], "timp pași [s]": run["time_steps_s"],
            "timp validare, toate epocile [s]": run["time_val_epochs_s"], "timp evaluare finală [s]": time_final,
            **cost_ledger(len(y_tr), run["epochs_run"], run["val_evals"], len(y_val)),
        }
        append_jsonl(RUNS_PATH, record)
        append_jsonl(HIST_PATH, {"seed": seed, "ratio": r, "history": run["history"]})
        done.add((seed, r))
        print(f"seed {seed}  1:{r}  n={len(y_tr):5d}  val AUPRC {metrics['AUPRC']:.3f}  epoca {run['best_epoch']:2d}/{run['epochs_run']}  "
              f"F1 {metrics['F1']:.3f}  timp {run['time_train_s']:.1f} s")
print(f"timp total al celulei: {time.perf_counter() - experiment_start:.1f} s")

## 9. Tabele

Toate valorile sunt pe **validation**, fără zgomot. Validation este folosit aici pentru dezvoltare: **epoca este aleasă pe el și pragul este optimizat pentru F1 tot pe el**, deci metricile sunt optimiste și **nu** sunt o estimare finală independentă. Estimarea finală se va face pe test, o singură dată, după ce alegem configurația.

In [ ]:
runs = pd.DataFrame([r for r in load_jsonl(RUNS_PATH, required=True) if r["seed"] in SEEDS and r["ratio"] in RATIOS])
runs = runs.sort_values(["seed", "ratio"]).reset_index(drop=True)
missing = sorted({(s, r) for s in SEEDS for r in RATIOS} - set(zip(runs["seed"], runs["ratio"])))
assert not missing, f"lipsesc rulări: {missing}"
n_seeds = runs["seed"].nunique()
is_smoke = MODE == "smoke"
smoke_note = "  [SMOKE TEST: verificare tehnică, nu concluzie comparativă]" if is_smoke else ""

detail_cols = ["seed", "raport", "fraude train", "legitime train", "total train", "val AUPRC", "val AUROC", "prag",
               "precision", "recall", "F1", "MCC", "TP", "FP", "FN", "TN", "epoca aleasă", "epoci executate",
               "pași Adam", "timp antrenare [s]", "timp pași [s]", "timp validare, toate epocile [s]", "timp evaluare finală [s]"]
print(f"Tabel detaliat (un rând per seed/raport) — {TABLE_LABEL}{smoke_note}")
display(runs[detail_cols].round(4))
runs[detail_cols].to_csv(RESULTS_DIR / "detailed_runs.csv", index=False)

In [ ]:
def mean_std(values, digits=3):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return f"{values.mean():.{digits}f} (std n/a: 1 seed)"
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


summary_rows = []
for r in RATIOS:
    part = runs[runs["ratio"] == r]
    for col in ["fraude train", "legitime train", "total train"]:
        assert part[col].nunique() == 1
    summary_rows.append({
        "model": f"VQC, raport 1:{r}", "raport fraude:legitime": f"1:{r}",
        "fraude train": int(part["fraude train"].iloc[0]), "legitime train": int(part["legitime train"].iloc[0]),
        "total train": int(part["total train"].iloc[0]),
        **{name: mean_std(part[col]) for name, col in [("val AUPRC", "val AUPRC"), ("val AUROC", "val AUROC"),
                                                         ("precision", "precision"), ("recall", "recall"), ("F1", "F1"), ("MCC", "MCC")]},
        "timp antrenare [s]": mean_std(part["timp antrenare [s]"], 1),
    })
summary = pd.DataFrame(summary_rows).set_index("model")
print(f"Tabel comparativ, medie ± std (ddof = 1) peste {n_seeds} seed(s) — {TABLE_LABEL}{smoke_note}")
if n_seeds < 2:
    print("Un singur seed: abaterea standard între seeds nu poate fi estimată (nu este zero).")
display(summary)
summary.to_csv(RESULTS_DIR / "summary_by_ratio.csv")

In [ ]:
cost_rows = []
for r in RATIOS:
    part = runs[runs["ratio"] == r]
    row = {"raport": f"1:{r}", "total train": int(part["total train"].iloc[0]),
           "epoci executate / rulare": mean_std(part["epoci executate"], 0) if part["epoci executate"].nunique() > 1 else int(part["epoci executate"].iloc[0]),
           "pași Adam / rulare": int(part["pași Adam"].iloc[0]) if part["pași Adam"].nunique() == 1 else mean_std(part["pași Adam"], 0)}
    for kind in ["training", "validation, toate epocile", "validation finală"]:
        per_run = part[f"exec {kind}"]
        row[f"exec {kind} / rulare"] = int(per_run.iloc[0]) if per_run.nunique() == 1 else f"{per_run.mean():,.0f} (variază)"
    row[f"exec training, total {n_seeds} rulări"] = int(part["exec training"].sum())
    row[f"exec validation (epoci + finală), total {n_seeds} rulări"] = int(part["exec validation, toate epocile"].sum() + part["exec validation finală"].sum())
    row[f"shots training, total {n_seeds} rulări"] = int(part["shots training"].sum())
    row[f"shots validation (epoci + finală), total {n_seeds} rulări"] = int(part["shots validation, toate epocile"].sum() + part["shots validation finală"].sum())
    cost_rows.append(row)
cost = pd.DataFrame(cost_rows).set_index("raport").astype(object)
total_row = {c: (cost[c].sum() if "total" in c and c != "total train" else "") for c in cost.columns}
cost.loc[f"TOTAL toate cele {len(runs)} rulări"] = total_row

print(f"Costul estimat pe hardware (nominal, parameter shift, {SHOTS_PER_EXECUTION} shots/execuție) — {TABLE_LABEL}{smoke_note}")
print("Coloanele '/ rulare' sunt per rulare (un seed, un raport); coloanele 'total' însumează toate seed-urile acelui raport.")
print("Nu sunt execuții reale ale simulatorului (backprop, shots=None).")
display(cost.map(lambda v: f"{v:,}" if isinstance(v, (int, np.integer)) else v))
cost.to_csv(RESULTS_DIR / "cost_ledger.csv")

## 10. Grafice

Punctele gri sunt seed-uri individuale. Marcajele colorate sunt media, iar barele de eroare arată ± std între seeds (doar când sunt ≥ 2 seeds).

In [ ]:
labels = [f"1:{r}" for r in RATIOS]
xpos = np.arange(len(RATIOS))
suptitle = TABLE_LABEL + (" · SMOKE TEST (verificare tehnică)" if is_smoke else f" · {n_seeds} seeds, {EPOCHS} epoci")


def per_ratio(col):
    groups = [runs.loc[runs["ratio"] == r, col].to_numpy(dtype=float) for r in RATIOS]
    means = np.array([g.mean() for g in groups])
    stds = np.array([g.std(ddof=1) if len(g) > 1 else np.nan for g in groups])
    return groups, means, stds


def plot_metric(ax, col, color, marker, label, offset=0.0):
    groups, means, stds = per_ratio(col)
    for i, g in enumerate(groups):
        ax.scatter(np.full(len(g), xpos[i] + offset), g, s=22, color=SEED_DOT_COLOR, alpha=0.7, zorder=2)
    ax.errorbar(xpos + offset, means, yerr=None if np.isnan(stds).all() else stds, color=color, marker=marker,
                markersize=8, linewidth=2, capsize=4, label=label, zorder=3)


fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_metric(axes[0], "val AUPRC", LEGIT_COLOR, "o", "medie")
axes[0].set_title("Validation AUPRC vs raport")
axes[0].set_ylabel("AUPRC (average precision)")

plot_metric(axes[1], "precision", LEGIT_COLOR, "o", "precision", -0.06)
plot_metric(axes[1], "recall", FRAUD_COLOR, "s", "recall", 0.06)
axes[1].set_title("Precision și recall la pragul max-F1")
axes[1].legend(loc="best")

_, train_exec, _ = per_ratio("exec training")
_, val_exec, _ = per_ratio("exec validation, toate epocile")
_, val_final, _ = per_ratio("exec validation finală")
width = 0.38
axes[2].bar(xpos - width / 2 - 0.01, train_exec / 1e6, width, color=LEGIT_COLOR, label="training (parameter shift)")
axes[2].bar(xpos + width / 2 + 0.01, (val_exec + val_final) / 1e6, width, color=FRAUD_COLOR, label="validation (epoci + finală)")
axes[2].set_title(f"Cost estimat per rulare (shots = execuții × {SHOTS_PER_EXECUTION})")
axes[2].set_ylabel("milioane de execuții / rulare")
axes[2].legend(loc="upper left")

for ax in axes:
    ax.set_xticks(xpos, labels)
    ax.set_xlabel("raport fraude : legitime (training)")
fig.suptitle(suptitle, fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "ratio_plots.png", dpi=130)
plt.show()

## 11. Interpretare

Celula următoare compară fiecare raport cu 1:1 **pe același seed** (diferențe pereche), pentru că între raporturi seed-ul fixează splitul, fraudele, legitimele 1:1, encoderul și ponderile inițiale. Numără în câte seeds diferența e pozitivă. Nu face test statistic formal: cu 5 seeds, puterea e mică.

Reguli de citire:
- **Smoke test:** numerele verifică doar că pipeline-ul merge cap-coadă. Două epoci și un seed nu spun nimic despre raporturi.
- O diferență se ia în serios doar dacă apare **în majoritatea seed-urilor** și este mare față de variația dintre seeds. Validation are doar 99 de fraude: mutarea unei singure fraude în clasament schimbă AUPRC vizibil.
- Rezultatele sunt pe validation, cu epoca și pragul alese tot pe validation, deci sunt optimiste pentru toate raporturile.
- Comparația e la **număr fix de epoci**, nu la buget fix: 1:5 face de ~3 ori mai mulți pași și costă de 3 ori mai mult la antrenare. Un câștig trebuie citit lângă costul din tabel.
- Encoderul e fitat pe 1:1. Concluzia se referă la „mai multe legitime cu reprezentare fixă”, nu la „mai multe legitime în general”.
- Lipsește încă referința clasică (logistic regression pe aceleași 4 unghiuri). Fără ea nu putem spune cât valorează rezultatul VQC.

In [ ]:
if is_smoke:
    print("SMOKE TEST: rezultatele de mai sus sunt doar verificări tehnice. Nu se trag concluzii comparative.")
    per_epoch = runs.groupby("ratio")["timp antrenare [s]"].mean() / runs.groupby("ratio")["epoci executate"].mean()
    estimate = per_epoch.sum() * 30 * 5
    print(f"Timp mediu per epocă (antrenare + validare): " + ", ".join(f"1:{r} {per_epoch[r]:.1f} s" for r in RATIOS))
    print(f"Estimare grosieră pentru experimentul complet (5 seeds × 3 raporturi × 30 epoci): ~{estimate / 60:.0f} min pe această mașină.")
else:
    base = runs[runs["ratio"] == 1].set_index("seed")
    rows = []
    for r in RATIOS[1:]:
        other = runs[runs["ratio"] == r].set_index("seed")
        for col in ["val AUPRC", "val AUROC", "precision", "recall", "F1", "MCC"]:
            diff = (other[col] - base[col]).loc[SEEDS]
            rows.append({"comparație": f"1:{r} − 1:1", "metrică": col, "diferență medie": diff.mean(), "std diferență": diff.std(ddof=1),
                         "seeds cu diferență > 0": f"{int((diff > 0).sum())} / {len(diff)}"})
    print(f"Diferențe pereche față de 1:1 (același seed) — {TABLE_LABEL}")
    display(pd.DataFrame(rows).round(4))
    print("Citire: o diferență medie mică față de std-ul ei, sau un semn care nu e consecvent între seeds, nu susține o concluzie.")

### Pași următori
1. Adaugă referința clasică (logistic regression pe aceleași unghiuri, același encoder per seed), ca rulare separată.
2. Doar pentru configurația aleasă la final: evaluare pe test cu `evaluate_under_noise` (p = 0, 0.001, 0.01, 0.05), pe 5 seeds.